# 03 — BART (abstractive)

**BART** (Lewis et al., 2019) è un transformer encoder-decoder pre-addestrato come denoising
autoencoder; il checkpoint usato qui, **`facebook/bart-large-cnn`**, è la versione fine-tuned per
la summarization sul dataset CNN/DailyMail (notizie single-document). A differenza di TextRank e
LexRank il riassunto è *generato* parola per parola, non estratto dal testo.

Questo notebook opera **solo sul campione condiviso** (`SCOPE` non esiste: la generazione
neurale sull'intero dataset non è praticabile). Su CPU contare ~30–90 s a esempio; con GPU NVIDIA
(rilevata automaticamente) pochi secondi a esempio.

**Limite noto**: l'input viene troncato ai primi **1024 token** (il massimo del modello). I
documenti Multi-News superano spesso questa soglia (mediana ~1.300 parole), quindi il modello
«vede» solo l'inizio del cluster di articoli — è la prassi standard per questi checkpoint e vale
allo stesso modo per PEGASUS (notebook 04), quindi il confronto resta equo.

Riassunti salvati incrementalmente in `results/summaries/bart_sample.tsv` con **ripresa** dopo
interruzione; metriche ricalcolabili dai soli file salvati.

In [ ]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer sentencepiece

In [1]:
# --- Configurazione ---------------------------------------------------------
import summ_utils as su

METODO     = 'bart'
SCOPE      = 'sample'    # questo notebook gira solo sul campione
N_SAMPLES  = 100         # deve combaciare con il campione creato dal notebook 00
SEED       = 42
LIMIT      = None        # es. 3 per uno smoke test rapido; None = tutti
CHECKPOINT = 'facebook/bart-large-cnn'
MAX_LEN    = 300         # lunghezza massima (token) del riassunto generato
NUM_BEAMS  = 4
MAX_INPUT  = 1024        # limite di input del modello

BASE   = su.trova_base_dir()
P      = su.percorsi_standard(BASE)
SAMPLE_PATH = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'
OUT_PATH    = P['summaries_dir'] / f'{METODO}_{SCOPE}.tsv'
DEVICE = su.rileva_device()

print(f'Checkpoint : {CHECKPOINT}')
print(f'Device     : {DEVICE}')
print(f'Output     : {OUT_PATH}')

Checkpoint : facebook/bart-large-cnn
Device     : cuda
Output     : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\summaries\bart_sample.tsv


## Generazione dei riassunti

La funzione `summ_ext_bart` di pyAutoSummarizer ricarica il modello **a ogni chiamata** e non usa
mai la GPU; qui replichiamo esattamente la sua logica (stessi parametri di generazione: beam
search a 4 beam, `early_stopping`, troncamento dell'input a 1024 token) ma caricando il modello
**una sola volta** e spostandolo sul `DEVICE` rilevato. Il separatore `` ||||| `` tra articoli
viene sostituito con un newline prima della tokenizzazione.

In [2]:
import torch
from transformers import BartForConditionalGeneration, BartTokenizer

tokenizer = BartTokenizer.from_pretrained(CHECKPOINT)
modello   = BartForConditionalGeneration.from_pretrained(CHECKPOINT).to(DEVICE).eval()

def genera(documento):
    inputs = tokenizer([documento], max_length=MAX_INPUT, truncation=True,
                       padding='longest', return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = modello.generate(inputs['input_ids'], attention_mask=inputs['attention_mask'],
                               num_beams=NUM_BEAMS, max_length=MAX_LEN, early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

esempi    = su.carica_campione(SAMPLE_PATH)
scrittore = su.ScrittoreRiassunti(OUT_PATH)
errori = su.ciclo_summarization(esempi, scrittore, genera, limit=LIMIT, etichetta='BART ')
scrittore.chiudi()

c:\Users\antonio.girasella\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\antonio.girasella\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\antonio.girasella\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate De

BART [1] media 1.4 s/esempio (saltati 3 gia' fatti)
BART [2] media 1.3 s/esempio (saltati 3 gia' fatti)
BART [3] media 1.2 s/esempio (saltati 3 gia' fatti)
BART [10] media 1.1 s/esempio (saltati 3 gia' fatti)
BART [20] media 1.1 s/esempio (saltati 3 gia' fatti)
BART [30] media 1.1 s/esempio (saltati 3 gia' fatti)
BART [40] media 1.0 s/esempio (saltati 3 gia' fatti)
BART [50] media 1.1 s/esempio (saltati 3 gia' fatti)
BART [60] media 1.1 s/esempio (saltati 3 gia' fatti)
BART [70] media 1.1 s/esempio (saltati 3 gia' fatti)
BART [80] media 1.1 s/esempio (saltati 3 gia' fatti)
BART [90] media 1.1 s/esempio (saltati 3 gia' fatti)
BART Completato: 97 nuovi, 3 saltati, 0 errori, 105 s totali


## Valutazione (indipendente dalla generazione)

Legge **solo** i file salvati; rieseguibile senza rigenerare i riassunti. Metriche ROUGE-1/2/L
(F1, precisione, recall), BLEU e METEOR con normalizzazione identica per tutti i metodi del
benchmark. Output: `results/metrics/bart_sample_per_example.csv` e `…_aggregate.json` (medie
complessive e per split).

In [3]:
import json

riassunti   = su.carica_riassunti(OUT_PATH)
riferimenti = su.carica_campione(SAMPLE_PATH)
config = {'checkpoint': CHECKPOINT, 'max_len': MAX_LEN, 'num_beams': NUM_BEAMS,
          'max_input_tokens': MAX_INPUT, 'device': DEVICE,
          'libreria': 'transformers (parametri di pyAutoSummarizer.summ_ext_bart)'}

righe, aggregato = su.valuta_e_salva(riferimenti, riassunti, METODO, SCOPE,
                                     P['metrics_dir'], config)
print(json.dumps(aggregato['overall'], indent=2))

Metriche per-esempio : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\bart_sample_per_example.csv (100 righe)
Metriche aggregate   : c:\Users\antonio.girasella\source\repos\multi-news-ai4stem-polito-master\results\metrics\bart_sample_aggregate.json
{
  "rouge1_f1": 0.2603076400556278,
  "rouge1_p": 0.5297044231142207,
  "rouge1_r": 0.1772640352016147,
  "rouge2_f1": 0.07168731882461947,
  "rouge2_p": 0.17253509334298653,
  "rouge2_r": 0.04660175734561311,
  "rougeL_f1": 0.13899929388636004,
  "rougeL_p": 0.340121233685227,
  "rougeL_r": 0.0900407736557004,
  "bleu": 0.01581061401679124,
  "meteor": 0.1636499885604578,
  "parole_generate": 55.78
}


## Ispezione qualitativa

In [4]:
su.mostra_esempi(su.carica_campione(SAMPLE_PATH), riassunti, quanti=2)

--- row_id=425 (split=train) ---
GENERATO   : Campaign mailer sent by GOP state Senate candidate Ed Charamut. Image of Democrat Matt Lesser, who is Jewish, with large, beady eyes, holding fistfuls of hundred-dollar bills. “A new line has been crossed,” said Rep. Lesser.
RIFERIMENTO: Just days after the slaying of 11 Jewish congregants at a Pittsburgh synagogue, a GOP candidate for a state Senate seat in Connecticut is accused of sending a mailer using an "age-old anti-Semitic trope." The ad sent out by Ed Charamut includes what the Washington Post calls a "money-grubbing" picture (here) of smiling opponent Matt Lesser, clutching $100 bills with a "crazed look in his eyes." Lesser says the original image of him was altered to add the cash and exaggerate his expression; Chara

--- row_id=1639 (split=train) ---
GENERATO   : Greenpeace activists scale the Shard, Europe's tallest skyscraper, as a protest against oil and gas drilling in the Arctic. The activists reached the top of the buildi